<a href="https://colab.research.google.com/github/naamasarshalom-art/segmentation_cellpose/blob/main/genotype_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Genotype Morphology Classification Pipeline

This notebook classifies cell nucleus images by genotype using a two-stage model pipeline.

## What it does
1. **Loads two pre-trained models:**
   - **Model 1** — RADIO-based SVM: separates trash (low-quality / non-classifiable images) from classifiable ones
   - **Model 2** — DINOv3-based logistic regression: classifies the remaining nuclei into *good*, *unclassified*, or *invaginated*

2. **Runs the pipeline over multiple batches**, each containing several genotype subfolders

3. **Saves results:**
   - A per-batch Excel file inside each batch folder
   - A single combined Excel file across all batches (`pipeline_results_all_batches.xlsx`)

## Output columns
| Column | Description |
|---|---|
| `batch` | Batch folder name |
| `genotype` | Sample ID |
| `genotype_name` | WT / Heterozygous / Homozygous mutant |
| `total_images` | Total images found |
| `n_trash` | Re

In [1]:
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ===================================================
# Block 1: Load RADIO model
# ===================================================
import torch
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor

device = "cuda"

model_version = "c-radio_v4-h"
model = torch.hub.load(
    'NVlabs/RADIO',
    'radio_model',
    version=model_version,
    skip_validation=True,
    force_reload=False
)
model.to(device).eval()
radio_model = model
print("✓ RADIO model loaded successfully")

/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/NVlabs/RADIO/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://huggingface.co/nvidia/C-RADIOv4-H/resolve/main/c-radio_v4-h_half.pth.tar?download=true" to /root/.cache/torch/hub/checkpoints/c-radio_v4-h_half.pth.tar


100%|██████████| 1.57G/1.57G [00:08<00:00, 187MB/s]
/root/.cache/torch/hub/NVlabs_RADIO_main/hubconf.py:80: UserWarning: Unexpected keys in state dict: ['patch_generator._vis_cond.norm_mean', 'patch_generator._vis_cond.norm_std']
  warnings.warn(f'Unexpected keys in state dict: {key_warn.unexpected_keys}')


✓ RADIO model loaded successfully


In [3]:
# ===================================================
# Block 2: Load Model 1 (RADIO-based SVM classifier)
# ===================================================
import joblib

scaler_m1 = joblib.load("/content/drive/MyDrive/model_nuc/model1_scaler.pkl")
clf_m1    = joblib.load("/content/drive/MyDrive/model_nuc/model1_svm.pkl")
print("✓ Model 1 loaded successfully")

✓ Model 1 loaded successfully


In [4]:
# ===================================================
# Block 3: Load DINO model
# ===================================================
from transformers import AutoModel, AutoImageProcessor
import torch

MODEL_DIR = "/content/drive/MyDrive/model_nuc/model_dinov3/dinov3_b16_weights"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

try:
    dino_model     = AutoModel.from_pretrained(MODEL_DIR).to(device).eval()
    dino_processor = AutoImageProcessor.from_pretrained(MODEL_DIR)
except Exception as e:
    print("Retrying with trust_remote_code=True")
    dino_model     = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True).to(device).eval()
    dino_processor = AutoImageProcessor.from_pretrained(MODEL_DIR, trust_remote_code=True)

print("✓ DINO model loaded successfully")

Running on: cuda


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

✓ DINO model loaded successfully


In [5]:
# ===================================================
# Block 4: Load Model 2 (DINO-based logistic regression)
# ===================================================
import joblib

scaler_m2 = joblib.load("/content/drive/MyDrive/model_nuc/model2_scaler.pkl")
pca_m2    = joblib.load("/content/drive/MyDrive/model_nuc/model2_pca.pkl")
clf_m2    = joblib.load("/content/drive/MyDrive/model_nuc/model2_clf.pkl")

print("✓ Model 2 loaded successfully")
print(f"  Classifier type: {type(clf_m2)}")

✓ Model 2 loaded successfully
  Classifier type: <class 'sklearn.linear_model._logistic.LogisticRegression'>


In [6]:
# ===================================================
# Block 5: Embedding functions — RADIO and DINO
# ===================================================
from tqdm.auto import tqdm

@torch.no_grad()
def get_radio_embeddings_batch(image_paths, batch_size=8):
    all_embs = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="RADIO", leave=False):
        batch_paths = image_paths[i:i+batch_size]
        pil_imgs    = [Image.open(p).convert("RGB") for p in batch_paths]
        w, h        = pil_imgs[0].size
        target_hw   = radio_model.get_nearest_supported_resolution(h, w)
        imgs = []
        for img in pil_imgs:
            img = img.resize((target_hw[1], target_hw[0]), Image.BILINEAR)
            imgs.append(pil_to_tensor(img).float() / 255.0)
        x = torch.stack(imgs).to(device)
        summary, _ = radio_model(x)
        all_embs.append(summary.cpu().numpy())
    return np.vstack(all_embs)

@torch.no_grad()
def get_dino_embeddings_batch(image_paths, batch_size=16):
    all_embs = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="DINO", leave=False):
        batch_paths = image_paths[i:i+batch_size]
        images  = [Image.open(p).convert("RGB") for p in batch_paths]
        inputs  = dino_processor(images=images, return_tensors="pt").to(device)
        outputs = dino_model(**inputs)
        cls     = outputs.last_hidden_state[:, 0, :]
        all_embs.append(cls.detach().cpu().numpy())
    return np.vstack(all_embs)

print("✓ Embedding functions ready")

✓ Embedding functions ready


In [7]:
# ===================================================
# Block 6: Batch configurations
# Edit SOURCE_DIR and GENOTYPES per batch, or run all at once below
# ===================================================
BATCHES = [
    {
        "source_dir": "/content/drive/MyDrive/model_nuc/nuclei_results_batch_1",
        "genotypes": {
            "63180": "WT",
            "98219": "Homozygous mutant",
            "98220": "Heterozygous",
            "98221": "Heterozygous"
        }
    },
    {
        "source_dir": "/content/drive/MyDrive/model_nuc/nuclei_results_batch_2",
        "genotypes": {
            "54795": "WT",
            "23037": "Homozygous mutant",
            "98219": "Homozygous mutant"
        }
    },
    {
        "source_dir": "/content/drive/MyDrive/model_nuc/nuclei_results_batch_3",
        "genotypes": {
            "54795": "WT",
            "23037": "Homozygous mutant",
            "98219": "Homozygous mutant",
            "98221": "Heterozygous"
        }
    },
    {
        "source_dir": "/content/drive/MyDrive/model_nuc/nuclei_results_batch_4",
        "genotypes": {
            "63180": "WT",
            "23037": "Homozygous mutant",
            "98219": "Homozygous mutant"
        }
    },
]
print(f"✓ {len(BATCHES)} batches configured")

✓ 4 batches configured


In [8]:
# ===================================================
# Block 7: Pipeline function — runs on one batch
# ===================================================
import os
import numpy as np
import pandas as pd
from pathlib import Path

def run_pipeline(source_dir, genotypes):
    results = []

    for genotype, geno_name in genotypes.items():
        folder = Path(source_dir) / genotype

        # Collect images — exclude with_bboxes
        img_paths = sorted([
            str(p) for p in
            list(folder.glob("*.jpg")) +
            list(folder.glob("*.png")) +
            list(folder.glob("*.tif"))
            if "with_bboxes" not in p.name.lower()
        ])

        if len(img_paths) == 0:
            print(f"  No images found in {genotype}")
            continue

        print(f"\n{'='*50}")
        print(f"Genotype: {genotype} ({geno_name}) — {len(img_paths)} images")

        # --- Model 1: RADIO → trash / classifiable ---
        X_radio  = get_radio_embeddings_batch(img_paths, batch_size=8)
        X_scaled = scaler_m1.transform(X_radio)
        m1_preds = clf_m1.predict(X_scaled)

        n_total        = len(img_paths)
        n_trash        = (m1_preds == 1).sum()
        n_classifiable = (m1_preds == 0).sum()

        print(f"  Model 1 → trash: {n_trash} | classifiable: {n_classifiable}")

        # --- Model 2: DINO → good / unclass / invaginated ---
        classifiable_idx   = np.where(m1_preds == 0)[0]
        classifiable_paths = [img_paths[i] for i in classifiable_idx]

        if len(classifiable_paths) == 0:
            print(f"  No classifiable images!")
            continue

        X_dino    = get_dino_embeddings_batch(classifiable_paths, batch_size=16)
        X_scaled2 = scaler_m2.transform(X_dino)
        X_pca2    = pca_m2.transform(X_scaled2)
        m2_proba  = clf_m2.predict_proba(X_pca2)[:, 1]

        m2_labels = np.where(m2_proba < 0.3, "good",
                    np.where(m2_proba > 0.8, "invaginated", "unclass"))

        n_good        = (m2_labels == "good").sum()
        n_unclass     = (m2_labels == "unclass").sum()
        n_invaginated = (m2_labels == "invaginated").sum()

        pct_good        = 100 * n_good        / n_classifiable
        pct_unclass     = 100 * n_unclass     / n_classifiable
        pct_invaginated = 100 * n_invaginated / n_classifiable

        print(f"  Model 2 → good: {n_good} | unclass: {n_unclass} | invaginated: {n_invaginated}")
        print(f"  % good of classifiable:        {pct_good:.1f}%")
        print(f"  % unclass of classifiable:     {pct_unclass:.1f}%")
        print(f"  % invaginated of classifiable: {pct_invaginated:.1f}%")

        results.append({
            "batch":                           Path(source_dir).name,
            "genotype":                        genotype,
            "genotype_name":                   geno_name,
            "total_images":                    n_total,
            "n_trash":                         n_trash,
            "n_classifiable":                  n_classifiable,
            "n_good":                          n_good,
            "n_unclass":                       n_unclass,
            "n_invaginated":                   n_invaginated,
            "pct_trash":                       round(100 * n_trash / n_total, 2),
            "pct_good_of_classifiable":        round(pct_good, 2),
            "pct_unclass_of_classifiable":     round(pct_unclass, 2),
            "pct_invaginated_of_classifiable": round(pct_invaginated, 2),
        })

    return results

print("✓ Pipeline function ready")

✓ Pipeline function ready


In [9]:
# ===================================================
# Block 8: Run all batches and save results
# - One Excel file per batch  (saved inside each batch folder)
# - One combined Excel file   (saved in model_nuc root)
# ===================================================

all_results = []

for batch in BATCHES:
    source_dir = batch["source_dir"]
    genotypes  = batch["genotypes"]
    batch_name = Path(source_dir).name

    print(f"\n{'#'*60}")
    print(f"Running batch: {batch_name}")
    print(f"{'#'*60}")

    batch_results = run_pipeline(source_dir, genotypes)
    all_results.extend(batch_results)

    # --- Save per-batch Excel ---
    if batch_results:
        df_batch = pd.DataFrame(batch_results)
        batch_out = Path(source_dir) / "pipeline_results.xlsx"
        with pd.ExcelWriter(batch_out, engine="openpyxl") as writer:
            df_batch.to_excel(writer, index=False, sheet_name="Results")
            ws = writer.sheets["Results"]
            for col in ws.columns:
                max_len = max(len(str(c.value)) if c.value is not None else 0 for c in col)
                ws.column_dimensions[col[0].column_letter].width = max_len + 4
        print(f"  Saved: {batch_out}")

# --- Save combined Excel ---
if all_results:
    df_all = pd.DataFrame(all_results)
    combined_out = Path("/content/drive/MyDrive/model_nuc") / "pipeline_results_all_batches.xlsx"
    with pd.ExcelWriter(combined_out, engine="openpyxl") as writer:
        df_all.to_excel(writer, index=False, sheet_name="All Results")
        ws = writer.sheets["All Results"]
        for col in ws.columns:
            max_len = max(len(str(c.value)) if c.value is not None else 0 for c in col)
            ws.column_dimensions[col[0].column_letter].width = max_len + 4
    print(f"\n✓ Combined results saved to: {combined_out}")
    print(f"  Total rows: {len(df_all)}")


############################################################
Running batch: nuclei_results_batch_1
############################################################

Genotype: 63180 (WT) — 316 images


RADIO:   0%|          | 0/40 [00:00<?, ?it/s]

  Model 1 → trash: 108 | classifiable: 208


DINO:   0%|          | 0/13 [00:00<?, ?it/s]

  Model 2 → good: 138 | unclass: 9 | invaginated: 61
  % good of classifiable:        66.3%
  % unclass of classifiable:     4.3%
  % invaginated of classifiable: 29.3%

Genotype: 98219 (Homozygous mutant) — 214 images


...

  Model 2 → good: 30 | unclass: 13 | invaginated: 60
  % good of classifiable:        29.1%
  % unclass of classifiable:     12.6%
  % invaginated of classifiable: 58.3%

Genotype: 98219 (Homozygous mutant) — 97 images


RADIO:   0%|          | 0/13 [00:00<?, ?it/s]

  Model 1 → trash: 34 | classifiable: 63


DINO:   0%|          | 0/4 [00:00<?, ?it/s]

  Model 2 → good: 4 | unclass: 2 | invaginated: 57
  % good of classifiable:        6.3%
  % unclass of classifiable:     3.2%
  % invaginated of classifiable: 90.5%
  Saved: /content/drive/MyDrive/model_nuc/nuclei_results_batch_4/pipeline_results.xlsx

✓ Combined results saved to: /content/drive/MyDrive/model_nuc/pipeline_results_all_batches.xlsx
  Total rows: 14
